# Sessions & Conversation State

*Notebook 17*

Give your agents conversation memory so they remember prior turns across separate runs.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, SQLiteSession

import asyncio

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

Without conversation state, each `Runner.run()` has no access to earlier turns.

A support bot forgets the ticket ID: a personal assistant forgets your preferences.

Sessions solve this.

---

## 🧠 Part 1: Without Sessions: The Forgetting Problem

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Be concise.",
    model=MODEL
)

# Turn 1

result1 = await Runner.run(agent, input="My name is Alex and I work in finance.")

print(f"Turn 1: {result1.final_output}\n")

# Turn 2: agent has no memory of Turn 1

result2 = await Runner.run(agent, input="What's my name and what do I do?")

print(f"Turn 2: {result2.final_output}")

The second run receives no history from Turn 1.

Any correct recall would be a guess.

---

## ✅ Part 2: With Sessions: Automatic Memory

Add `session=session` to `Runner.run()`.

The SDK loads past history and persists new items for you.

The new input can be stored before the run completes.

Use one stable session ID per conversation.

Prefer a chat or user-plus-thread ID: a bare user ID can merge separate conversations.

With no `db_path`, history lives in that session object's in-memory database, so pass the same object each turn.

Part 3 uses `db_path` so a new session object can reconnect later.

In [ ]:
# Create a session: in-memory by default
session = SQLiteSession("demo_session")

# Turn 1

result1 = await Runner.run(agent, input="My name is Alex and I work in finance.", session=session)

print(f"Turn 1: {result1.final_output}\n")

# Turn 2: agent remembers Turn 1

result2 = await Runner.run(agent, input="What's my name and what do I do?", session=session)

print(f"Turn 2: {result2.final_output}\n")

# Turn 3: agent remembers both previous turns

result3 = await Runner.run(agent, input="Based on my background, what AI tools might be useful for me?", session=session)

print(f"Turn 3: {result3.final_output}")

### 💡 Key Takeaway

`session=session` lets the SDK carry conversation history between runs.

---

## 💾 Part 3: Persistent Sessions: Surviving Restarts

In-memory sessions are lost when the kernel restarts.

A file-based session writes history to a SQLite file.

The same conversation is available next time you open the notebook.

⚠️ **Security note:** Session files contain conversation history: protect them as user data.

In [ ]:
# File-based session: persists across kernel restarts
persistent_session = SQLiteSession(
    "persistent_demo",
    db_path="demo_sessions.sqlite"
)

result = await Runner.run(agent, input="My favorite programming language is Python and I prefer concise answers.", session=persistent_session)

print(f"Turn 1: {result.final_output}")
print(f"\n💾 History saved to: demo_sessions.sqlite")

#### Reconnect with a New Session Object

In [ ]:
# Recreate the session with the same ID and path.
# This is the setup an app would use after a restart.
reconnected_session = SQLiteSession(
    "persistent_demo",
    db_path="demo_sessions.sqlite"
)

# Prove the saved preference reconnected before relying on the model to recall
reconnected_items = await reconnected_session.get_items()
stored_text = " ".join(
    str(item.get("content", ""))
    for item in reconnected_items
    if item.get("role") == "user"
)
history_reconnected = "Python" in stored_text and "concise" in stored_text

print(
    f"History reconnected: {'yes' if history_reconnected else 'no'} "
    f"({len(reconnected_items)} items)"
)

result = await Runner.run(agent, input="What do you remember about my preferences?", session=reconnected_session)

print(f"After reconnect: {result.final_output}")

### 💡 Key Takeaway

The session ID and `db_path` together identify the conversation store.

As long as both match, any new session object connects to the same history.

This works even after a kernel restart.

---

## 🤝 Part 4: Shared Sessions Across Agents

A session isn't tied to one agent. It's a conversation store.

Multiple agents can share the same session, each seeing the full conversation history.

In [ ]:
# Two specialist agents sharing one session

onboarding_agent = Agent(
    name="OnboardingAgent",
    instructions="You help new users get started. Collect their name, role, and goals.",
    model=MODEL
)

advisor_agent = Agent(
    name="AdvisorAgent",
    instructions="You give personalized advice based on the user's background and goals.",
    model=MODEL
)

shared_session = SQLiteSession("shared_demo")

# --------------------------------------------------------------
print("✅ Shared session ready")

#### Onboarding Collects Context

In [ ]:
# Onboarding agent collects user info

result = await Runner.run(onboarding_agent, input="Hi, I'm Jordan. I'm a marketing manager trying to learn AI tools.", session=shared_session)

print(f"Onboarding: {result.final_output}")

#### Advisor Uses the Context

In [ ]:
# Advisor agent sees the full conversation: knows who Jordan is

result = await Runner.run(advisor_agent, input="What should I focus on first?", session=shared_session)

print(f"Advisor: {result.final_output}")

### 💡 Key Takeaway

Both agents read from and write to the same conversation history.

Give separately run agents the same session.

They continue one conversation without manual state passing.

---

## 🧹 Part 5: Managing Session State

Sessions expose methods to inspect and control their history.

Use `get_items()` to read the stored items: message items have `role` and `content`, other item types differ.

Use `clear_session()` when a user starts a new chat or you want to drop stale context.

In [ ]:
# Inspect the session history
items = await shared_session.get_items()
print(f"📋 Session history: {len(items)} items stored")

# Inspect stored session items
for item_number, item in enumerate(items, 1):
    role = item.get("role", "unknown")
    content = str(item.get("content", ""))[:80]
    print(f"  {item_number}. [{role}] {content}...")

#### Clear a Session

In [ ]:
# Clear a session to start fresh
await shared_session.clear_session()

items_after = await shared_session.get_items()
print(f"✅ Session cleared: {len(items_after)} items remaining")

---

## 🧹 Demo Cleanup

Removes the demo's session database so the exercises start clean.

In [ ]:
# Close open connections before deleting (open SQLite files can lock on some systems)
for db_session in (persistent_session, reconnected_session):
    await asyncio.to_thread(db_session.close)

# Remove the demo SQLite database and its WAL sidecar files
for suffix in ("", "-wal", "-shm"):
    Path(f"demo_sessions.sqlite{suffix}").unlink(missing_ok=True)

# --------------------------------------------------------------
print("🧹 Cleanup complete")

---

## 💪 Practice Exercises

### Exercise 1: Personal Study Assistant

*Covers: persistent sessions, reconnecting to saved history*

Build a study assistant that remembers completed topics after you recreate its session.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Personal Study Assistant
# --------------------------------------------------------------
# Objective: Build an assistant that remembers covered topics.

# TODO 1: Create a StudyAssistant agent with instructions to:
#           - Track which topics the student has covered
#           - Suggest what to study next based on history
#           - Answer questions about previously covered material

# TODO 2: Create a file-backed SQLiteSession: store the db_path in a variable

# TODO 3: Run two turns:
#           Turn 1: "I just finished studying Python basics and functions"
#           Turn 2: "What should I study next?"
#           Print each response

# TODO 4: Recreate the session with the same ID and db_path

# TODO 5: Prove the history reconnected: call get_items() on the recreated
#           session and confirm the original topic is present BEFORE asking.
#           Then ask "What have I covered so far?" and print the response

# TODO 6: Close the sessions with asyncio.to_thread(session.close), then delete
#           the db_path file plus its -wal and -shm sidecars

# --- Write your code below this line ---

### Exercise 2: Multi-Turn Interview Bot

*Covers: conversation state: building multi-turn memory*

Build an interview bot that asks questions one at a time and remembers all previous answers.

**Hint:** starter instructions:

> Ask one focused question at a time.
> After the third answer, summarize what you learned.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Multi-Turn Interview Bot
# --------------------------------------------------------------
# Objective: An interviewer that builds context across turns.

# TODO 1: Create an InterviewBot agent with instructions to:
#           - Ask one focused question at a time
#           - Reference previous answers when asking follow-ups
#           - After 3 questions, summarize what it has learned

# TODO 2: Create an in-memory SQLiteSession

# TODO 3: Run 4 turns simulating an interview:
#           Turn 1: Start the interview (the bot asks its first question)
#           Turn 2: Answer the first question
#           Turn 3: Answer the follow-up question
#           Turn 4: Answer the third question and ask for a summary
#           Print each response

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Sessions store conversation history:**

- `SQLiteSession("id")`: in-memory, lost on restart

- `SQLiteSession("id", db_path="file.sqlite")`: persistent, survives restarts

- Pass the same session to every run that should share its history
<br>
<br>

**Sessions are shareable:**

- Multiple agents can use the same session: all see the full conversation history

- This lets separately run agents continue the same conversation
<br>
<br>

**Sessions are controllable:**

- `get_items()`: inspect the history

- `clear_session()`: deletes all conversation items while keeping the session object
<br>
<br>

**A session preserves a transcript, not a curated memory:**

- Notebook 18 shows how to keep durable facts and drop noise

---

## 📍 Next Step

**Notebook 18: Persistent Memory**  

Turn conversation history into curated memory.

Keep durable facts and preferences, and learn what not to store.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-17-sessions-and-conversation-state)

---